# Superstore Profitability, Customer, and Geographic Analysis

This notebook is the reproducible analytical foundation for the accompanying Power BI dashboard and controlling report. It analyses profitability by product, discount tier, customer segment, geography, and customer account.

> **Unit-of-analysis note:** The raw dataset contains transaction-line records. Product, category, sub-category, discount-tier, and line-profit analyses use transaction lines. Order count and AOV use distinct `Order ID` values. Customer profitability aggregates all transaction lines by `Customer ID`.


## 1. Imports and data loading

The loading logic supports both Kaggle and a local GitHub project. Download the CSV from Kaggle and store it at `data/Sample - Superstore.csv` for local execution.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

LOCAL_DATA_PATH = Path('../data/Sample - Superstore.csv')
KAGGLE_DATA_PATH = Path(
    '/kaggle/input/datasets/vivek468/superstore-dataset-final/'
    'Sample - Superstore.csv'
)

if KAGGLE_DATA_PATH.exists():
    DATA_PATH = KAGGLE_DATA_PATH
    environment = 'Kaggle'
elif LOCAL_DATA_PATH.exists():
    DATA_PATH = LOCAL_DATA_PATH
    environment = 'Local / GitHub project'
else:
    raise FileNotFoundError(
        'Dataset not found. Attach the Kaggle dataset or place ' 
        'Sample - Superstore.csv in ../data/.'
    )

df = pd.read_csv(DATA_PATH, encoding='latin1')
print(f'Environment: {environment}')
print(f'Loaded data from: {DATA_PATH}')
df.head()


## 2. Data quality and feature engineering


In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Month Number'] = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.month_name()
df['Year'] = df['Order Date'].dt.year

quality_report = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum(),
    'unique_values': df.nunique()
})
display(quality_report)

data_checks = pd.Series({
    'transaction_lines': len(df),
    'unique_orders': df['Order ID'].nunique(),
    'unique_customers': df['Customer ID'].nunique(),
    'exact_duplicate_rows': df.duplicated().sum(),
    'negative_shipping_days': (df['Shipping Days'] < 0).sum(),
    'discount_outside_0_to_1': (~df['Discount'].between(0, 1)).sum(),
    'first_order_date': df['Order Date'].min(),
    'last_order_date': df['Order Date'].max()
}, name='value')
display(data_checks)

assert df['Sales'].notna().all(), 'Sales contains missing values'
assert df['Profit'].notna().all(), 'Profit contains missing values'
assert df['Discount'].between(0, 1).all(), 'Discount must be between 0 and 1'
assert (df['Shipping Days'] >= 0).all(), 'Ship Date precedes Order Date'


## 3. Analytical tables and reusable metrics


In [ ]:
order_df = (
    df.groupby('Order ID', as_index=False)
      .agg(
          order_date=('Order Date', 'min'),
          sales=('Sales', 'sum'),
          profit=('Profit', 'sum'),
          quantity=('Quantity', 'sum'),
          average_discount=('Discount', 'mean'),
          segment=('Segment', 'first'),
          region=('Region', 'first')
      )
)

customer_df = (
    df.groupby(['Customer ID', 'Customer Name'], as_index=False)
      .agg(
          sales=('Sales', 'sum'),
          profit=('Profit', 'sum'),
          orders=('Order ID', 'nunique'),
          average_discount=('Discount', 'mean'),
          segment=('Segment', 'first')
      )
)
customer_df['profit_margin_pct'] = 100 * customer_df['profit'] / customer_df['sales']

def summarize_performance(data, group_column):
    summary = (
        data.groupby(group_column, as_index=False)
            .agg(
                sales=('Sales', 'sum'),
                profit=('Profit', 'sum'),
                average_discount=('Discount', 'mean'),
                transaction_lines=('Order ID', 'size'),
                orders=('Order ID', 'nunique')
            )
    )
    summary['profit_margin_pct'] = 100 * summary['profit'] / summary['sales']
    return summary

category_summary = summarize_performance(df, 'Category').sort_values('profit', ascending=False)
subcategory_summary = summarize_performance(df, 'Sub-Category').sort_values('profit')
region_summary = summarize_performance(df, 'Region').sort_values('profit', ascending=False)

headline_metrics = pd.Series({
    'transaction_lines': len(df),
    'unique_orders': order_df['Order ID'].nunique(),
    'unique_customers': customer_df['Customer ID'].nunique(),
    'total_sales': df['Sales'].sum(),
    'total_profit': df['Profit'].sum(),
    'profit_margin_pct': 100 * df['Profit'].sum() / df['Sales'].sum(),
    'average_order_value': order_df['sales'].mean(),
    'loss_making_line_rate_pct': 100 * (df['Profit'] < 0).mean(),
    'loss_making_customers': (customer_df['profit'] < 0).sum()
}, name='value')
display(headline_metrics)


## 4. Category and sub-category profitability

Technology and Office Supplies are the largest profit contributors, while Furniture has a substantially lower margin. At aggregate sub-category level, the Power BI model identifies Tables, Bookcases, and Supplies as negative-profit sub-categories.


In [ ]:
display(category_summary)
display(subcategory_summary)

plot_data = subcategory_summary.sort_values('profit')
bar_colors = np.where(plot_data['profit'] < 0, '#D64550', '#E85AAD')

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(plot_data['Sub-Category'], plot_data['profit'], color=bar_colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit by Sub-Category')
ax.set_xlabel('Profit ($)')
plt.tight_layout()
plt.show()

furniture_summary = (
    subcategory_summary[
        subcategory_summary['Sub-Category'].isin(
            ['Chairs', 'Furnishings', 'Bookcases', 'Tables']
        )
    ]
    .sort_values('profit', ascending=False)
)
display(furniture_summary)


## 5. Discount-tier profitability and two loss measures

This section distinguishes: (1) sub-categories with negative aggregate profit and (2) sub-categories contributing the largest loss dollars among loss-making transaction lines. These are complementary, not interchangeable, measures.


In [ ]:
discount_summary = (
    df.groupby('Discount', as_index=False)
      .agg(
          transaction_lines=('Order ID', 'size'),
          mean_profit=('Profit', 'mean'),
          median_profit=('Profit', 'median'),
          total_profit=('Profit', 'sum')
      )
      .sort_values('Discount')
)

discount_profit_correlation = df['Discount'].corr(df['Profit'])
print(f'Discount–profit correlation: {discount_profit_correlation:.3f}')
display(discount_summary)

fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.bar(discount_summary['Discount'], discount_summary['mean_profit'], width=0.035, color='#E85AAD')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.axvline(0.20, color='#F0C419', linewidth=1.2, linestyle='--')
ax1.axvline(0.30, color='#D64550', linewidth=1.2, linestyle='--')
ax1.set_xlabel('Discount rate')
ax1.set_ylabel('Average transaction-line profit ($)')
ax1.set_title('Average Profit by Discount Tier')

ax2 = ax1.twinx()
ax2.plot(discount_summary['Discount'], discount_summary['transaction_lines'], color='#3FA7D6', marker='o')
ax2.set_ylabel('Transaction lines')
plt.tight_layout()
plt.show()

negative_aggregate_subcategories = subcategory_summary[subcategory_summary['profit'] < 0].copy()
display(negative_aggregate_subcategories)

loss_lines = df[df['Profit'] < 0].copy()
loss_line_by_subcategory = (
    loss_lines.groupby('Sub-Category', as_index=False)
              .agg(loss_dollars=('Profit', 'sum'), transaction_lines=('Order ID', 'size'))
              .sort_values('loss_dollars')
)
loss_line_by_subcategory['loss_dollars_abs'] = -loss_line_by_subcategory['loss_dollars']
loss_line_by_subcategory['loss_share_pct'] = (
    100 * loss_line_by_subcategory['loss_dollars_abs']
    / loss_line_by_subcategory['loss_dollars_abs'].sum()
)

top_loss_line_subcategories = loss_line_by_subcategory.head(4).copy()
display(top_loss_line_subcategories)
print(
    'Top-four loss-line sub-category share: '
    f'{top_loss_line_subcategories["loss_share_pct"].sum():.1f}%'
)


## 6. Regression: conditional discount association

The OLS model uses HC3 heteroskedasticity-robust standard errors. Its coefficient is interpreted as an association conditional on category, region, and segment—not a causal effect.


In [ ]:
base_model = smf.ols(
    'Profit ~ Discount + C(Category) + C(Region) + C(Segment)',
    data=df
).fit(cov_type='HC3')

interaction_model = smf.ols(
    'Profit ~ Discount * C(Category) + C(Region) + C(Segment)',
    data=df
).fit(cov_type='HC3')

coefficient_table = pd.DataFrame({
    'coefficient': base_model.params,
    'robust_se': base_model.bse,
    'p_value': base_model.pvalues,
    'ci_lower': base_model.conf_int()[0],
    'ci_upper': base_model.conf_int()[1]
})

display(coefficient_table)
print(f'Base-model R-squared: {base_model.rsquared:.3f}')
print(f'Discount coefficient: {base_model.params["Discount"]:.2f}')
print(
    'Association of a +10 percentage-point discount increase: '
    f'{0.10 * base_model.params["Discount"]:.2f} profit dollars per transaction line'
)
print(base_model.summary())


## 7. Segment, seasonal, region, city, and customer analysis


In [ ]:
segment_summary = summarize_performance(df, 'Segment').sort_values('profit', ascending=False)
city_summary = summarize_performance(df, 'City').sort_values('sales', ascending=False)

display(segment_summary)
display(region_summary)
display(city_summary.head(10))

monthly_summary = (
    df.set_index('Order Date')
      .resample('ME')
      .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
      .reset_index()
)
monthly_summary['profit_margin_pct'] = 100 * monthly_summary['profit'] / monthly_summary['sales']

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly_summary['Order Date'], monthly_summary['sales'], color='#F0C419', label='Sales')
ax1.set_ylabel('Sales ($)')
ax2 = ax1.twinx()
ax2.plot(monthly_summary['Order Date'], monthly_summary['profit'], color='#E85AAD', label='Profit')
ax2.set_ylabel('Profit ($)')
ax1.set_title('Monthly Sales and Profit')
plt.tight_layout()
plt.show()

q4_data = df[df['Month Number'].isin([9, 10, 11, 12])].copy()
segment_month_sales = q4_data.groupby(['Month Number', 'Segment'])['Sales'].sum().unstack()
display(segment_month_sales)

consumer_q4_orders = (
    df[(df['Segment'] == 'Consumer') & (df['Month Number'].isin([9, 10, 11]))]
      .groupby(['Month Number', 'Order ID'], as_index=False)['Sales']
      .sum()
      .groupby('Month Number', as_index=False)
      .agg(order_count=('Order ID', 'nunique'), average_order_value=('Sales', 'mean'), total_sales=('Sales', 'sum'))
)
consumer_q4_orders['order_count_change_pct'] = 100 * consumer_q4_orders['order_count'].pct_change()
consumer_q4_orders['aov_change_pct'] = 100 * consumer_q4_orders['average_order_value'].pct_change()
display(consumer_q4_orders)


## 8. Customer profitability deep dive


In [ ]:
customer_df = customer_df.sort_values('profit', ascending=False).copy()
customer_df['cumulative_profit_pct'] = 100 * customer_df['profit'].cumsum() / customer_df['profit'].sum()

customer_metrics = pd.Series({
    'customers_analyzed': customer_df['Customer ID'].nunique(),
    'loss_making_customers': (customer_df['profit'] < 0).sum(),
    'top_10_customer_profit_share_pct': 100 * customer_df.head(10)['profit'].sum() / customer_df['profit'].sum()
}, name='value')
display(customer_metrics)

fig, ax = plt.subplots(figsize=(12, 7))
segment_colors = {'Consumer': '#E85AAD', 'Corporate': '#A61C9C', 'Home Office': '#45278C'}
for segment, segment_data in customer_df.groupby('segment'):
    ax.scatter(
        segment_data['sales'],
        segment_data['profit'],
        s=25 + 12 * segment_data['orders'],
        alpha=0.65,
        color=segment_colors.get(segment, '#E85AAD'),
        label=segment
    )
ax.axhline(0, color='#F0C419', linestyle='--', linewidth=1.2)
ax.set_title('Customer Sales vs Profit')
ax.set_xlabel('Customer Sales ($)')
ax.set_ylabel('Customer Profit ($)')
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

loss_making_customers = customer_df[customer_df['profit'] < 0].sort_values('profit')
display(loss_making_customers.head(20))


## 9. Export reconciliation tables for Power BI

The exported files can be used to reconcile Dashboard, report, and notebook outputs.


In [ ]:
overall_reconciliation = headline_metrics.reset_index()
overall_reconciliation.columns = ['metric', 'value']

subcategory_reconciliation = subcategory_summary.copy()
customer_reconciliation = customer_df.copy()

overall_reconciliation.to_csv('overall_reconciliation.csv', index=False)
subcategory_reconciliation.to_csv('subcategory_reconciliation.csv', index=False)
customer_reconciliation.to_csv('customer_reconciliation.csv', index=False)

print('Saved: overall_reconciliation.csv')
print('Saved: subcategory_reconciliation.csv')
print('Saved: customer_reconciliation.csv')


## 10. Interpretation and limitations

- The sample dataset does not provide separate COGS, shipping cost, marketing spend, promotion cost, or inventory cost.
- The discount regression is associational; it should not be interpreted as a causal estimate.
- The October diagnosis identifies the Consumer segment and the order-volume/AOV mechanics, not an external cause.
- Before publication, run **Restart Session and Run All** to ensure the notebook is reproducible without missing-variable errors.
